# Shekina Flow: AlchemistPrime + Registry + DaathGraph

Este cuaderno demuestra:
- Transformaciones de AlchemistPrime con una configuración típica (ASSIGN_CONSTANT, ENRICH_WITH_JOIN, CLEAN_IDENTIFIER, HOMOLOGATE, DATETIME_BUILDER, CLEAN_ID_NUMERICO, CONDITIONAL_MAPPING).
- Orquestación con Shekina y un Registry mínimo: mirror.select_df → alchemist.transmute → mirror.upsert_df.
- Validación rápida de DaathGraph en memoria (agregar nodos/aristas).

In [ ]:
# Imports y setup
import json, os, sys
import pandas as pd
from typing import Dict, Any
from pathlib import Path

# Asegurar que el directorio raíz del repo (que contiene 'src') esté en sys.path
here = Path.cwd()
repo_root = None
for p in [here, *here.parents]:
    if (p / 'src').is_dir():
        repo_root = str(p)
        break
if repo_root and repo_root not in sys.path:
    sys.path.append(repo_root)
print('repo_root:', repo_root)

# Garantizar dependencias clave antes de importar módulos del proyecto
import subprocess

def ensure(pkg_import: str, pip_name: str = None):
    try:
        __import__(pkg_import)
        print(f"{pkg_import} (pre-check): OK")
    except Exception as e:
        print(f"{pkg_import} no disponible, instalando con pip...", e)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name or pkg_import])
        __import__(pkg_import)
        print(f"{pkg_import} (post-install): OK")

# Polars (para AlchemistPrime)
ensure('polars', 'polars')
import polars as pl
print('polars version:', pl.__version__)

# SQLAlchemy (usado por conectores del join)
ensure('sqlalchemy', 'SQLAlchemy')
from sqlalchemy import __version__ as sa_version
print('sqlalchemy version:', sa_version)

# psycopg2 (necesario para DaathGraph aunque no usemos BD en este demo)
try:
    import psycopg2  # type: ignore
    print('psycopg2 (pre-check): OK')
except Exception as e:
    print('psycopg2 no disponible, instalando psycopg2-binary...', e)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'psycopg2-binary'])
    import psycopg2  # type: ignore
    print('psycopg2 (post-install): OK')

from src.alchemist.alchemist_prime import AlchemistPrime
from src.shekina.shekina import Shekina
from src.daath.daath_graph import DaathGraph

print('Versions: pandas', pd.__version__)

repo_root: c:\shekina
polars (pre-check): 1.33.1
polars (pre-check): 1.33.1


ModuleNotFoundError: No module named 'sqlalchemy'

## Datos de ejemplo (stp11__citas_fin_ajustada) y diccionario de homologación
Simulamos un subconjunto mínimo para probar todas las transformaciones.

In [ ]:
# Datos base (citas) y diccionario
df_data = pd.DataFrame([
    {"NOM_UNEG": "Sede Principal 01", "fecha_asigna": "2025-01-15", "hora_asigna": "13:45", "ID_CITA": "CITA-001", "CODIGO_CUPS": "CUP-123", "ambito_cups": "Consulta Externa", "codConsulta": "C123"},
    {"NOM_UNEG": "sede principal 01", "fecha_asigna": "2025/01/15", "hora_asigna": "13:45", "ID_CITA": "CITA-002", "CODIGO_CUPS": "CUP-999", "ambito_cups": "Domiciliaria", "codConsulta": "C999"}
])

# Diccionario de homologación (concepto 'sede')
df_dict = pd.DataFrame([
    {"concepto_normal": "sede", "valor_normal": "sede_principal_01", "es_sinonimo_de": "8000061506"}
])

df_data

## Tabla de configuración de AlchemistPrime
Incluye todos los tipos de paso requeridos en el orden descrito.

In [ ]:
# Configuración del flujo (equivalente a la tabla solicitada)
df_flow = pd.DataFrame([
    {"row_id":1, "step_id":10, "source_table":"stp11__citas_fin_ajustada", "source_column":None, "step_type":"ASSIGN_CONSTANT", "step_params":json.dumps({"value": "8000061506"}), "target_column":"numDocumentoIdObligado"},
    {"row_id":2, "step_id":20, "source_table":"stp11__citas_fin_ajustada", "source_column":None, "step_type":"ENRICH_WITH_JOIN", "step_params":json.dumps({
        "source": {"source_type": "csv", "path": "CUPSGrServicios.csv"},
        "left_on": ["codConsulta"], "right_on": ["Extra_I:CUPS"],
        "select": {"Extra_II:GrServicios": "codServicio"}, "how": "left"}), "target_column":"join_codServicio"},
    {"row_id":3, "step_id":30, "source_table":"stp11__citas_fin_ajustada", "source_column":"NOM_UNEG", "step_type":"CLEAN_IDENTIFIER", "step_params":json.dumps({}), "target_column":"nom_uneg_clean"},
    {"row_id":4, "step_id":40, "source_table":"stp11__citas_fin_ajustada", "source_column":"nom_uneg_clean", "step_type":"HOMOLOGATE", "step_params":json.dumps({"dictionary_key": "sede"}), "target_column":"codPrestador"},
    {"row_id":5, "step_id":50, "source_table":"stp11__citas_fin_ajustada", "source_column":"fecha_asigna", "step_type":"DATETIME_BUILDER", "step_params":json.dumps({"time_col": "hora_asigna", "format": "%Y-%m-%d %H:%M"}), "target_column":"fechaInicioAtencion"},
    {"row_id":6, "step_id":60, "source_table":"stp11__citas_fin_ajustada", "source_column":"ID_CITA", "step_type":"CLEAN_ID_NUMERICO", "step_params":json.dumps({}), "target_column":"numAutorizacion"},
    {"row_id":7, "step_id":70, "source_table":"stp11__citas_fin_ajustada", "source_column":"CODIGO_CUPS", "step_type":"CLEAN_ID_NUMERICO", "step_params":json.dumps({}), "target_column":"codConsulta"},
    {"row_id":8, "step_id":80, "source_table":"stp11__citas_fin_ajustada", "source_column":"ambito_cups", "step_type":"CLEAN_IDENTIFIER", "step_params":json.dumps({}), "target_column":"ambito_cups_clean"},
    {"row_id":9, "step_id":90, "source_table":"stp11__citas_fin_ajustada", "source_column":"ambito_cups_clean", "step_type":"CONDITIONAL_MAPPING", "step_params":json.dumps({"rules":[{"if":{"operator":"contains", "value":"domiciliaria"}, "then":"03"}], "default":"01"}), "target_column":"modalidadGrupoServicioTecSal"},
    {"row_id":10, "step_id":100, "source_table":"stp11__citas_fin_ajustada", "source_column":None, "step_type":"ASSIGN_CONSTANT", "step_params":json.dumps({"value": "01"}), "target_column":"grupoServicios"}
])
df_flow

## Ejecutar AlchemistPrime.transmute()
Cubre todas las transformaciones anteriores. Si el CSV del join no existe, crea uno mínimo temporal.

In [ ]:
# Preparar archivo CSV de enriquecimiento si no existe
csv_path = 'CUPSGrServicios.csv'
if not os.path.exists(csv_path):
    pd.DataFrame([{"Extra_I:CUPS": "C123", "Extra_II:GrServicios": "S001"}]).to_csv(csv_path, index=False)

alchemist = AlchemistPrime(data_df=df_data, config_df=df_flow, dictionary_df=df_dict)
alchemist.transmute()
df_result = alchemist.to_pandas()
df_result.head()

## Orquestación con Shekina + Registry (mirror/select → alchemist/transmute → mirror/upsert)
Mostramos cómo una fila puede invocar una transmutación completa.

In [ ]:
# Registry mínimo de ejemplo
class MiniRegistry:
    def __init__(self):
        
        self._map = {}
    def register(self, opcode, fn):
        
        self._map[opcode] = fn
    def resolve(self, opcode):
        
        return self._map.get(opcode)

# Simuladores de Mirror y Alchemist en Registry
def mirror_select_df(context: Dict[str, Any], payload: Dict[str, Any]):
        
    # Aquí normalmente leerías de Postgres/CSV/Excel. Usamos df_data y df_flow del notebook.
    return {"df_data": df_data.copy(), "df_flow": df_flow.copy(), "df_dict": df_dict.copy()}

def alchemist_transmute(context: Dict[str, Any], payload: Dict[str, Any]):
        
    dd = context.get('mirror.select', payload.get('inputs', [{}]))
    data = dd.get('df_data'); flow = dd.get('df_flow'); dicc = dd.get('df_dict')
    alq = AlchemistPrime(data_df=data, config_df=flow, dictionary_df=dicc)
    alq.transmute().agregar_status_y_errores().limpiar_columnas_intermedias()
    return {"df_data_transmuted": alq.to_pandas()}

def mirror_upsert_df(context: Dict[str, Any], payload: Dict[str, Any]):
        
    # En un caso real, haríamos UPSERT en la BD. Aquí solo devolvemos el DataFrame para inspección.
    res = context.get('alchemist.transmute', {})
    return {"upserted": res.get('df_data_transmuted')}

reg = MiniRegistry()
reg.register('mirror.select_df', lambda ctx, p: {'mirror.select': mirror_select_df(ctx, p)})
reg.register('alchemist.transmute', lambda ctx, p: {'alchemist.transmute': alchemist_transmute(ctx, p)})
reg.register('mirror.upsert_df', lambda ctx, p: {'mirror.upsert': mirror_upsert_df(ctx, p)})

# Tabla canónica mínima para orquestar la transmutación
process_table = [
    {"id": "s1", "step": "mirror.read", "opcode": "mirror.select_df", "payload": {} , "outputs": ["mirror.select"]},
    {"id": "s2", "step": "alchemist.run", "opcode": "alchemist.transmute", "depends_on": ["s1"], "payload": {}, "outputs": ["alchemist.transmute"]},
    {"id": "s3", "step": "mirror.upsert", "opcode": "mirror.upsert_df", "depends_on": ["s2"], "payload": {"table": "demo.out", "keys": ["ID_CITA"]}, "outputs": ["mirror.upsert"]}
]

shekina = Shekina(config={})
context, log = shekina.run_process_table(process_table, registry=reg)
print('Run log:', log)
context.get('alchemist.transmute', {}).get('df_data_transmuted').head()

## Mini demo de DaathGraph (en memoria)
Validamos que la clase funciona: crear nodos/aristas sin tocar BD.

In [ ]:
dg = DaathGraph(db_params=None)
a = dg.build_uri('sispro', 'CUPS', 'C123')
b = dg.build_uri('servicio', 'Servicio', 'S001')
dg.add_node('CUPS', a, 'Consulta demo')
dg.add_node('Servicio', b, 'Servicio demo')
dg.add_edge(a, b, 'PERTENECE_A')
len_nodes = dg.graph.number_of_nodes(); len_edges = dg.graph.number_of_edges()
len_nodes, len_edges

## 🛠️ Prompt de diagnóstico para errores de importación/dependencias

Cuando veas un error como `ModuleNotFoundError: No module named 'polars'` o `No module named 'src'`, usa este checklist/prompt:

1) Inspeccionar estructura del proyecto (árbol mínimo)
- ¿Existe una carpeta `src/` en el raíz del repo?
- ¿El notebook se ejecuta dentro del repo? Si no, añade el raíz que contenga `src` a `sys.path`.

2) Verificar entorno de Python activo
- ¿Qué Python está usando el notebook? Muestra `sys.executable` y `sys.version`.
- ¿Lista de paquetes instalada? Busca el paquete faltante (e.g. `polars`).

3) Instalar dependencias que faltan
- Preferente: conda/mamba con el environment.yml del repo (con `conda-forge`).
- En notebooks: `pip install paquete` como fallback si conda no está disponible.

4) Reproducir ruta de importación
- Imprime `sys.path` y valida que incluya la raíz del repo.
- Intenta `import src.<modulo>...` manualmente y captura el error con `traceback`.

5) Validar versiones mínimas
- Si la importación falla por ABI/cambios de versión, pinna una versión estable conocida (ej. `polars>=1.0`).

6) Automatizar fix en notebooks
- Envuelve el `import polars` en un try/except que haga `pip install polars` si falta (ya implementado arriba).

7) Si el error persiste
- Busca el import dentro del código fuente que lo requiere (`grep "import polars" -R src/`).
- Confirmar que no haya imports circulares o rutas relativas incorrectas.

8) Para producción/CI
- Asegura que `environment.yml` incluya canales (ej. `conda-forge`) y dependencias; o añade `polars` en `pyproject`/`requirements.txt`.
